In [1]:
import numpy as np
import pandas as pd

import os 
import sys
sys.path.append("..//utils/")
import color_utils
import process_SC_data
from load_matlab_data import loadmat_sbx
sys.path.append("..//neural/")
from format_behavior_data import load_behavior_data, get_feeder_ints, get_feeder_periods, classify_feeder_ints
sys.path.append("..//stim/")
from format_chronic_stim import idx_cells_by_stim

import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

In [2]:
root_dir = "Z:/Isabel/data/Grid Caching Data/"
save_figs_dir = f"../figures/basic_neural_analysis_sc/"
session_info_file = f"Z:/Isabel/data/hpc_implants/good_sessions.xlsx"

In [3]:
''' Bird and session list '''
data_dict = {}

all_dirs = sorted(os.listdir(root_dir))
ignore_files = ['arena_im_1_1.mat', '.DS_Store']
session_dirs = []
for s_dir in all_dirs:
    if s_dir in ignore_files:
        continue
    elif '_' in s_dir:
        session_dirs.append(s_dir)

bird_ids = []
for session_folder in session_dirs:
    parts = session_folder.split('_')
    bird = parts[0]
    session_id = f'{parts[1]}_{parts[2]}'
    if bird in bird_ids:
        data_dict[bird][session_id] = {}
    else:
        data_dict[bird] = {}
        data_dict[bird][session_id] = {}
        bird_ids.append(bird)

In [4]:
''' Data params '''
bird = 'SLV143' # update as needed
fps = 60 # Hz
dt = 1/fps

# collect sessions with pose tracking & ephys
behavior_sessions = data_dict[bird].keys()

In [6]:
# get feeder open/close times
session_info = pd.read_excel(session_info_file, sheet_name=bird, header=1)
session_info["id"] = session_info["date"].dt.strftime("%y%m%d")
sessions_to_use = []
for i, b in enumerate(behavior_sessions):
    print(b[:6])
    if any(b[:6] == session_info["id"]):
        sessions_to_use.append(b)

230209
230213
230215
230217
230220
230222
230224
230227
230301
230303
230306
230308
230313
230315
230317


In [7]:
session_id = sessions_to_use[0]

In [8]:
data_dir = f"{root_dir}{bird}_{session_id}/"

In [9]:
seed_struct, count_data = load_behavior_data(data_dir)

Z:/Isabel/data/Grid Caching Data/SLV143_230224_141441/annotatedSeeds.mat


In [10]:
# get all perch interactions
all_perch_start = count_data['newPerch']
all_perch_end = count_data['endPerch']
all_perch_idx = count_data['perchNum']
n_perches = all_perch_start.shape[0]

In [12]:
np.unique(all_perch_idx)

array([ -4,  -3,  -2,  -1,   1,   2,   3,   4,   5,   8,   9,  10,  11,
        12,  13,  14,  15,  16,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118,
       119, 120, 121, 122, 123, 124, 125, 126, 127, 128], dtype=int16)

In [12]:
# get feeder open/close times
session_info = pd.read_excel(session_info_file, sheet_name=bird, header=1)
session_info["id"] = session_info["date"].dt.strftime("%y%m%d")
feeder_times_raw = session_info.loc[session_info["id"] == session_id[:6],
                                               "feeder open times"].iloc[0]

IndexError: single positional indexer is out-of-bounds

In [7]:
feeder_times_raw

nan

In [7]:
feeder_ranges = feeder_times_raw.split(sep=', ')
feeder_open_list = []
feeder_close_list = []
for fr in feeder_ranges:
    times = fr.split(sep='-')
    feeder_open_list.append(times[0])
    feeder_close_list.append(times[1])
feeder_open_times = [int(t) for t in feeder_open_list]
feeder_close_times = [int(t) for t in feeder_close_list]

In [11]:
feeder_times_raw = '10-22'

In [12]:
feeder_ranges = feeder_times_raw.split(sep=', ')
feeder_open_list = []
feeder_close_list = []
for fr in feeder_ranges:
    times = fr.split(sep='-')
    feeder_open_list.append(times[0])
    feeder_close_list.append(times[1])
feeder_open_times = [int(t) for t in feeder_open_list]
feeder_close_times = [int(t) for t in feeder_close_list]

In [14]:
feeder_close_times

[22]